In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import torch
sys.path.append('../../')

In [3]:
from neural_control.dynamics import SequentialDualSourcingModel

In [4]:
N = 2
T = 5

sds = SequentialDualSourcingModel(lr=3, le=2)
sds.I_0 = torch.tensor(10.0)


In [5]:
sds.reset(N)
inital_inventories = sds.I_0.repeat(N, 1)
initial_demands = sds.all_demands[0]
initial_qr = torch.cat(sds.previous_qr, dim=-1)
initial_qe = torch.cat(sds.previous_qe, dim=-1)

future_demands = torch.randint(low=1, high=4, size=(N, T))
demands = torch.cat([initial_demands, future_demands], dim=-1)

new_qe = torch.randn([N, T]).abs().round()
new_qr = torch.randn([N, T]).abs().round()

qr = torch.cat([initial_qr, new_qr], dim=-1) if sds.lr > 0 else new_qr
qe = torch.cat([initial_qe, new_qe], dim=-1) if sds.le > 0 else new_qe

qe_arrived = qe[:, :T]
qr_arrived = qr[:, :T]

qe_ordered = qe[:, sds.le:]
qr_ordered = qr[:, sds.lr:]

costs, invs = sds.replay_multisteps(inital_inventories, 
                                                qra=qr_arrived,
                                                qea=qe_arrived,
                                                qro=qr_ordered,
                                                qeo=qe_ordered,
                                                all_demands=demands
                                               )

In [6]:
demands[0, 1:]

tensor([3., 3., 1., 2., 3.])

In [7]:
qr_arrived[0]

tensor([0., 0., 0., 0., 1.])

In [8]:
qe_arrived[0]

tensor([0., 0., 3., 0., 2.])

In [9]:
# multistep
invs[0]

tensor([7., 4., 6., 4., 4.])

In [10]:
step_wise_costs[0]

NameError: name 'step_wise_costs' is not defined

In [11]:
all_costs = []
all_invs = []
sds.reset(N)
previous_inventory = sds.I_0.repeat(N)

for i in range(T):
    current_demand = demands[:, i+1]
    qra = qr_arrived[:, i]
    qea = qe_arrived[:, i]
    qr_i = qr_ordered[:, i]
    qe_i = qe_ordered[:, i]
    cs, previous_inventory = sds.replay_step(
                    previous_inventory, 
                    current_demand, 
                    qra=qra, 
                    qea=qea,
                    qr=qr_i,
                    qe=qe_i
                   )
    all_invs.append(previous_inventory)
    all_costs.append(cs)
step_wise_invs = torch.stack(all_invs, -1)[:, :]
step_wise_costs = torch.stack(all_costs, -1)

In [12]:
demands[0, 1:]

tensor([3., 3., 1., 2., 3.])

In [13]:
step_wise_invs[0, 1:]

tensor([4., 6., 4., 4.])

In [14]:
step_wise_invs[:, :] - invs[:, :]

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

In [15]:
costs - step_wise_costs

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

In [ ]:
step_wise_costs - costs

In [ ]:
invs - step_wise_invs[:, 1:]

In [ ]:
step_wise_invs - invs

In [ ]:
invs

In [ ]:
torch.stack(all_costs, dim=-1) 

In [ ]:
invs